### Notebook to fix the design fasta
- Postprocess the bam file (found duplicates)
  - if mapping quality ($5) >= 1 
		- check identity (~95% of sequence should be matched) with cigar and md
		- optional: expected length of the read (configuration) ~>265
	- if mapping quality ($5) < 0:
		- check identity (~95% of sequence should be matched) with cigar and md
		- if mapping on forward ($2 == 0)
			- check if AS > XS (check how many cases have difference of 5) -> keep and set mapping quality ($5) = 1 (is the best alignment)
		- if mapping with reversed read ($2 == 16) and strandedness (configuration) is set:
			- try recovering / finding the correct match
			  - if AS == XS (the line should have a XA)
					- it can be recovered if only one of the alternatives at XA (separated by ";") can be matched on +strand (XA:Z:<oligo_name>,+/-<pos>,<cigar>,<number_differences>;)
					- if more than one alternative can be matched on + than we count and throw a warning of the overall number

In [ ]:
import pysam
import pandas
import sys
import yaml

config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/config/config.yml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

# 

In [ ]:
# helpful functions
def determine_MD_NM(differences,sequence,alnLength):
  NM = 0
  MD = ''
  lpos = -1
  for refPos,seqPos,edit in differences:
    if type(edit[1]) != type('T'): 
      NM+=edit[1]
      if edit[0] == 'INS': pass
      else:
        if lpos+1 <= refPos:
          MD += "%d^%s"%(refPos-(lpos+1),edit[2])
          lpos = refPos-1+edit[1]
        else:
          MD += "^%s"%(edit[2])
          lpos += edit[1]
    else:
      if lpos+1 <= refPos:
        MD += "%d%s"%(refPos-(lpos+1),edit[1])
      else:
        MD += "%s"%(edit[1])
      lpos = refPos+len(edit[1])-1
      NM += len(edit[1])
  if lpos+1 <= alnLength:
    MD += "%d"%(alnLength-(lpos+1))
  return MD,NM


def aln_length(cigarlist):
  tlength = 0
  for operation,length in cigarlist:
    if operation == 0 or operation == 2 or operation == 3 or operation >= 6: tlength += length
  return tlength


def expected_length_filter(read, expected_length):
  """Filter reads that are of an expected length (AS tag is used because match gives 1 => if AS >= expected_length, then read is of expected length and quality)"""
  if read.get_tag('AS') >= expected_length:
    return True
  return False


def calculate_sequence_identity(read):
  """Compute sequence identity form a read using cigar string, MD tag, and NM tag"""
  cigar = read.cigarstring
  MD = read.get_tag('MD')
  NM = read.get_tag('NM')
  alnLength = aln_length(read.cigartuples)
  return (alnLength - NM)/alnLength, NM


def get_number_of_matches_per_strand(read):
  """Get number of matches per forward and reverse strand assuming read has XA tag"""
  # split ";" on right side of XA tag
  xa_tag = read.get_tag('XA').rstrip(';')
  xa_list = xa_tag.split(';')
  forward_matches = 0
  reverse_matches = 0
  for xa in xa_list:
    if '+' in xa.split(",")[-3]: # because oligo name might include ";"
      forward_matches += 1
    elif '-' in xa.split(",")[-3]:
      reverse_matches += 1
    else:
      sys.stderr.write("Error: XA tag does not contain strand information")
      sys.exit()
  return forward_matches, reverse_matches


def get_XA_information(read):
  """Select the XA information from the alternative alignment on the positive strand"""
  try:
    prepare_xa = read.get_tag("XA").split(",") # oligo name might include ";"
    reference_name = prepare_xa[0]
    next_elem = prepare_xa[1:] # remove first name
  except KeyError:
    sys.stderr.write("Error: No XA tag found")
    return -1
  while len(next_elem) > 0:
      next_elem = ",".join(next_elem) # join back
      next_elem = next_elem.split(";") # join back, split on ; in order to find one element
      elem = next_elem[0]
      if "+" in elem.split(",")[-3]:
          return f"{reference_name},{elem}"
      # join again with one less
      next_elem = ";".join(next_elem[1:])
      prepare_xa = next_elem.split(",")
      reference_name = prepare_xa[0]
      next_elem = prepare_xa[1:] # remove first name
  sys.stderr.write("Error: No forward alignment found in XA tag")
  return -1


def print_read_information(read):
  """Debugging and showing read information to the user"""
  print("Reference: ", read.reference_name)
  print("Cigar info: ", read.cigarstring, read.cigartuples)
  sequence_identity, num_mismatches = calculate_sequence_identity(read)
  print("Identity and mismatch", sequence_identity, num_mismatches)
  print("flag: ", read.flag)
  print("reference start: ", read.reference_start)
  print("mapping quality: ", read.mapping_quality)
  print("Tag information", read.tags) 


def get_barcode(read):
  """The barcode is stored in XI tag:XI:Z:<barcode>,YI:I:<unknown>"""
  # check if it does not include "N" 
  barcode = read.get_tag("XI").split(",")[0]
  if "N" in barcode.upper():
      sys.stderr.write("Error: Barcode of {read.query_name}")
      return "failed"
  return barcode


def prepare_table_information(read, case="normal"):
  """Prepares the information of the association:
    @case: 
      - normal: the information from the alignment are taken
      - fix_mapping_quality: the mapping quality is set to 1 but all information is taken from the alignment
      - rescue: we rescue a better alignment from the XA tag
  """
  barcode = get_barcode(read)
  reference_name = read.reference_name
  position=read.reference_start # 0-based
  cigarstring=read.cigarstring
  nm=read.get_tag("NM")
  md=read.get_tag("MD")
  if case == "normal":
    mapping_quality = read.mapping_quality
  
  if case == "fix_mapping_quality":
    mapping_quality = 1
    
  if case == "rescue":
    # if NM is 0, then we can rescue the alignment from XA tag
    xa_info = get_XA_information(read)
    reference_name = xa_info.split(",")[0]
    position = int(xa_info.split(",")[1])
    cigarstring = xa_info.split(",")[2]
    nm = int(xa_info.split(",")[3])
    md = read.get_tag("MD")
    mapping_quality = 1
    if nm != 0:
      md = "unknown" # if NM != 0: we don't know MD tag
  return f"{barcode}\t{reference_name}\t{position};{cigarstring};NM:i:{nm};MD:Z:{md};{mapping_quality}"

In [ ]:
# use split as test data 
identity_threshold = config["general"]["identity_threshold"]
mismatches_threshold = config["general"]["mismatches_threshold"]
use_expected_alignment_length = config["general"]["use_expected_alignment_length"]
expected_alignment_length = config["general"]["expected_alignment_length"]
# code from /data/gpfs-1/users/kisa11_c/work/coding/MPRA/bin/removeSequenceErrors.py
bamfile = config["files"]["example_bam"]
bamfile = config["files"]["merged_bam"]

input_file = pysam.Samfile( bamfile, "rb" )
# open output_bamfile for writing
# output_file = pysam.Samfile( output_bamfile, "wb", template=input_file )
count = 0
high_mismatches_count = 0
high_identity_count = 0
unmapped_count = 0
reversed_read_count = 0
rescued_read_count = 0
not_rescuable_count = 0
count_no_second_alignment = 0
count_low_identity_but_high_score = 0
no_xi_count = 0
change_quality_count = 0
high_quali_reversed_read_count = 0
high_quality_same_as_xs = 0
high_quality_alignment = 0
low_quality_alignment = 0
show_example = True
for read in input_file:
  count += 1
  # if count > 10: sys.exit() # debug
  # # check if NOS3 in reference name (debug)
  # if "NOS3" not in read.reference_name: continue
  #### Counting of specific read properties
  # check if all alignments have XI tag
  try:
    # print(read.get_tag("XI"))
    xi = read.get_tag("XI")
  except:
    no_xi_count += 1 # 0 xi count in sample data

  # check if read has low identity but high alignment score
  sequence_identity, num_mismatches = calculate_sequence_identity(read)
  if sequence_identity < identity_threshold:
    if read.get_tag('AS') > 265:
      count_low_identity_but_high_score += 1
  
  
  # skip reads with certain properties
  if read.is_unmapped:
    unmapped_count += 1
    continue # because of weird line (NB501960:812:HH53WAFX5:1:11101:13917:2374	4	*	0	0	None	*	0	0	GGTG)

  if not sequence_identity >= identity_threshold: continue # TODO: put identity_threshold in config
  # high alignment identity => potentially interesting alignments
  high_identity_count += 1

  if num_mismatches > mismatches_threshold: # throw warning # TODO: put number of allowable mismatches in config
    high_mismatches_count += 1
    # print("WARNING: number of missmatches from %s %d"%(read.query_name, num_mismatches))
  
  # filter for expected sequence length
  if use_expected_alignment_length:
    if not expected_length_filter(read,expected_alignment_length): continue
  
  
  # modify mapping quality for reads on forward with AS > XS
  if read.mapping_quality < 1:
    low_quality_alignment += 1
    if read.flag == 0: # check if it is a best alignment
      if read.get_tag("AS") > read.get_tag("XS"):
        change_quality_count += 1
        prepare_table_information(read, case="fix_mapping_quality")

    # rescue reads with reversed read (flag == 16) with AS = XS and check if only on other alignment on forward strand is given
    if read.flag == 16:
      reversed_read_count += 1
      if read.get_tag("AS") == read.get_tag("XS"):
        try: 
          read.get_tag("XA")
        except:
          sys.stderr.write("WARNING: read (%s) can not be rescued because XA tag is not given"%(read.query_name))
          count_no_second_alignment += 1
          continue
        # check if only one alignment on forward strand is given
        forward_matches, reverse_matches = get_number_of_matches_per_strand(read)
        if forward_matches == 1: # best alignment found on forward strand
          # change alignment to forward strand with the information from XA tag
          prepare_table_information(read, case="rescue")
          rescued_read_count += 1
        elif forward_matches > 1:
          not_rescuable_count += 1
          #Throw warning that the read can not be rescued
          sys.stderr.write("WARNING: read (%s) can not be rescued because more than one alignment on forward strand is given"%(read.query_name))
  
  else: # mapping quality >= 1
    # normal case
    prepare_table_information(read, case="normal")
    high_quality_alignment += 1
    # check if reversed read has AS = XS
    if read.flag == 16:
      high_quali_reversed_read_count += 1
      if read.get_tag("AS") == read.get_tag("XS"):
        # if get_tag("XA") is not None
        high_quality_same_as_xs += 1
        try:
          read.get_tag("XA")
        except:
          # sys.stderr.write("WARNING: read (%s) can not be rescued because XA tag is not given"%(read.query_name))
          count_no_second_alignment += 1
          continue  
        

print("unmapped count: ", unmapped_count)
print("reversed_read_count: ", reversed_read_count)
print("reversed rescued_read_count: ", rescued_read_count)
print("It is a best alignment: ", change_quality_count)
print("All rescued reads: ", change_quality_count + rescued_read_count)
print("not_rescuable_count: ", not_rescuable_count)
print("high_identity_count: ", high_identity_count)
print("high_mismatches_count: ", high_mismatches_count)
print("count: ", count)
print("proportion of high quality: ", high_identity_count/count)
print("high mapping quality bwa: ", high_quality_alignment)
print("low mapping quality bwa: ", low_quality_alignment)
print("count_no_second_alignment: ", count_no_second_alignment)
print("count_low_identity_but_high_score: ", count_low_identity_but_high_score)
print("No xi count: ", no_xi_count)
print("high_quali_reversed_read_count: ", high_quali_reversed_read_count)
print("high_quality_same_as_xs: ", high_quality_same_as_xs)

In [ ]:
# use split as test data 
identity_threshold = 0.98
mismatches_threshold = 3
consider_sequence_length = True
expected_sequence_length = 265
# code from /data/gpfs-1/users/kisa11_c/work/coding/MPRA/bin/removeSequenceErrors.py
bamfile = config["files"]["example_bam"]
output_bamfile = config["files"]["example_bam"] + ".filtered"
input_file = pysam.Samfile( bamfile, "rb" )
# open output_bamfile for writing
output_file = pysam.Samfile( output_bamfile, "wb", template=input_file )
count = 0
high_mismatches_count = 0
high_identity_count = 0
unmapped_count = 0
reversed_read_count = 0
rescued_read_count = 0
not_rescuable_count = 0
count_no_second_alignment = 0
count_low_identity_but_high_score = 0
show_example = True
for read in input_file:
  # if "NOS3" not in read.reference_name: continue
  count += 1
  # if count > 10: sys.exit() # debug
  if read.is_unmapped:
    unmapped_count += 1
    continue # because of weird line (NB501960:812:HH53WAFX5:1:11101:13917:2374	4	*	0	0	None	*	0	0	GGTG)
  sequence_identity, num_mismatches = calculate_sequence_identity(read)
  if sequence_identity < identity_threshold:
    # print information of read 
    # check alignment score if > 265 count_low_identity_but_high_score += 1
    if read.get_tag('AS') > 265:
      count_low_identity_but_high_score += 1
    
    # count += 1
  if not sequence_identity > identity_threshold: continue # TODO: put identity_threshold in config
  high_identity_count += 1

  
  if num_mismatches > mismatches_threshold: # throw warning # TODO: put number of allowable mismatches in config
    high_mismatches_count += 1
    # print("WARNING: number of missmatches %d"%(num_mismatches))
  
  #### TODO: ADD here the expected sequence filter
  if consider_sequence_length:
    if not expected_length_filter(read,expected_sequence_length): continue
  # TODO: if mapping quality 0 => AS = XS => Check XA how many 
  # check if NOS3 in reference name (debug)
  
  # modify mapping quality for reads on forward with AS > XS
  if read.mapping_quality < 1:
    if read.flag == 0:
      if read.get_tag("AS") > read.get_tag("XS"):
        read.mapping_quality = 1

    # TODO: rescue reads with reversed read (flag == 16) with AS = XS and check if only on other alignment on forward strand is given
    if read.flag == 16:
      reversed_read_count += 1
      if read.get_tag("AS") == read.get_tag("XS"):
        # check if get_tag("XA") is not None
        try: 
          read.get_tag("XA")
        except:
          sys.stderr.write("WARNING: read can not be rescued because XA tag is not given")
          # print(read)
          # print(read.reference_name)
          count_no_second_alignment += 1
          continue
          
        # check if only one alignment on forward strand is given
        forward_matches, reverse_matches = get_number_of_matches_per_strand(read)
        if forward_matches == 1: # best alignment found on forward strand
          # TODO: change alignment to forward strand with the information from XA tag
          rescued_read_count += 1
          read = get_forward_alignment_from_xa(read)
        elif forward_matches > 1:
          not_rescuable_count += 1
          # #Throw warning that the read can not be rescued
          # sys.stderr.write("WARNING: read can not be rescued because more than one alignment on forward strand is given")
          
          
          
        # print(read.get_tag("XA")) # cardiac_neuro_cava_random:KCNH2|ENSG00000055118.17|EH38E3808220_rev_tile1-1,+16,270M,0;
        # count += 1
        # if 
  # output_file.write(read)
  
  
  
  #   # print(read)
  #   # print("mapping quality: ", read.mapping_quality)
  #   # print(read.tags)
  #   # print(read.reference_name)
  #   # check if AS == XS
  #   if read.get_tag("AS") == read.get_tag("XS"):
  #     print(read)
  #     print(read.get_tag("AS"))
  #     print(read.get_tag("XA"))
    

  # rstart = read.reference_start # 0-based
  # MDval = None
  # NMval = None
  # new = []
  # for key,value in read.tags:
  #   if key == "MD":
  #     MDval = value
  #   elif key == "NM":
  #     NMval = value
  #   else:
  #     new.append((key,value))
  # estart,eend = 0,0
  # res = filter(lambda x: True if (type(x[2][1]) != type('T')) else ((estart > x[0]+rstart) or (x[0]+rstart > eend)),parseMDwithCigar(MDval,read.cigartuples,read.query_sequence))
  # print("interesting res: ", list(res))

  # MDval,NMval = determine_MD_NM(res,read.query_sequence,aln_length(read.cigartuples))
  # read.tags = [("NM",NMval),("MD",MDval)]+new
  # # print(type(MDval), MDval)
  # if MDval != '270':
  #     count += 1
  # print("interesting_determine_MD_NM: MD: %s NM: %s new: %s"%(MDval, NMval, new))
  # print("important: ", read.tags)
print("unmapped count: ", unmapped_count)
print("reversed_read_count: ", reversed_read_count)
print("rescued_read_count: ", rescued_read_count)
print("not_rescuable_count: ", not_rescuable_count)
print("high_identity_count: ", high_identity_count)
print("high_mismatches_count: ", high_mismatches_count)
print("count: ", count)
print("proportion of high quality: ", high_identity_count/count)
print("count_no_second_alignment: ", count_no_second_alignment)
print("count_low_identity_but_high_score: ", count_low_identity_but_high_score)
# unmapped count:  762
# reversed_read_count:  16927
# rescued_read_count:  0
# not_rescuable_count:  0
# high_identity_count:  3663761
# high_mismatches_count:  79523
# count:  3804439
# proportion of high quality:  0.9630226690452915
# count_no_second_alignment:  8887

#### Old approach try to fix the design fasta (not good, because it goes simpler with postprocessing the bam file)
- This notebook focuses on finding duplicates
  - prepare new fasta file with reverse and forward sequences
```bash
design_fasta=/fast/groups/ag_kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header.fa
forw_revc_fasta=/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/fixed_design/design_no_duplicates_sequence_and_header_forw_revc.fa
paste <(
    cat $design_fasta | awk '{{if ($1 ~ /^>/) {{ gsub(/[\]\[]/,"_"); print ">"substr($1,2)"_forw"}}}}';
    cat $design_fasta | awk '{{if ($1 ~ /^>/) {{ gsub(/[\]\[]/,"_"); print ">"substr($1,2)"_revc"}}}}';
) <(
    cat $design_fasta | awk '{{if ($1 ~ /^[^>]/) {{ seq=seq$1}}; if ($1 ~ /^>/ && NR!=1) {{print seq; seq=""}}}} END {{print seq}}';
    cat $design_fasta | awk '{{if ($1 ~ /^[^>]/) {{ seq=substr($1,16,270)}}; if ($1 ~ /^>/ && NR!=1) {{print seq; seq=""}}}} END {{print seq}}' | tr ACGTacgt TGCAtgca | rev;
) > $forw_revc_fasta
```
  - Merge duplicated headers (attention not too long headers)
- Other problems to long headers (add_match_tbl_gc_kircher.ipynb)


In [ ]:
import pandas as pd
import yaml

In [ ]:
def write_fasta(df, output_file, columns=["header", "sequence"]):
    """
    Write a dataframe to a fasta file
    @param df: dataframe to write
    @param output_file: output file name
    @param columns: columns to write to file (expected two strings (first is header, second is sequence))
    """
    # only keep the columns we want
    df = df[columns]
    print(df.head())
    # write df as fasta to file
    with open(output_file, "w") as handle:
        for index, row in df.iterrows():
            handle.write(row[columns[0]] + "\n")
            handle.write(row[columns[1]] + "\n")
    return output_file

def merge_duplicated_headers(duplicated_sequences_dict):
    """merge the headers of duplicated sequences to one header and returns list of merged headers"""
    # ### implemented logic: 
    # if same label, write the label in the beginning and merge the second header without the label add _revcomp to the end of the second header
    # if not the same label, write both headers with the label and add _revcomp to the end of the second header
    header_list = []
    for key, value in duplicated_sequences_dict.items():
        # check if same label
        if len(value) == 2:
            # if same label, merge headers
            if value[0].split(':')[0] == value[1].split(':')[0]:
                label=value[0].split(':')[0]
                new_header=label + ":" + "_".join(":".join(value[0].split(':')[1:]).split("_")[:-1]) + "_" + "_".join(":".join(value[0].split(':')[1:]).split("_")[:-1]) + "_revcomp" # add _revcomp to the end of the second header
            else: # different labels
                new_header="_".join(value[0].split("_")[:-1]) + "_" + "_".join(value[1].split("_")[:-1]) + "_revcomp" # add _revcomp to the end of the second header
        # add the new header
        header_list.append(new_header)
    return header_list

In [ ]:
# load config from /data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/config/config.yml
config_path = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/config/config.yml'

with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [ ]:
# load forward and reverse complement reference sequences 
forw_revcomp_ref_df = pd.read_csv(config["mapping"]["forward_revcomp_reference_tbl"], sep='\t', header=None, names=['header', 'sequence'])

# find if sequences are duplicated
forw_revcomp_ref_df.sequence.duplicated().sum() # 1802

# write to fasta and check with unix command
write_fasta(forw_revcomp_ref_df, config["mapping"]["forward_revcomp_reference"])

In [ ]:
# check for duplicated sequences in forw_revcomp_ref_df
forw_revcomp_ref_df.sequence.duplicated().sum() # 1802

# find headers with duplicated sequences
duplicated_sequences = forw_revcomp_ref_df[forw_revcomp_ref_df.duplicated(subset='sequence', keep=False)]

# iterate duplicated_sequences data frame and create dictionary with sequences as keys and list of headers with that sequence as values
duplicated_sequences_dict = {}
for index, row in duplicated_sequences.iterrows(): # usage of iterrows is fine, because table only has 1802 rows
    if row['sequence'] in duplicated_sequences_dict:
        duplicated_sequences_dict[row['sequence']].append(row['header'])
    else:
        duplicated_sequences_dict[row['sequence']] = [row['header']]

# check number of values per key
number_duplicates = {}
for key, value in duplicated_sequences_dict.items():
    if len(value) in number_duplicates:
        number_duplicates[len(value)] += 1
    else:
        number_duplicates[len(value)] = 1
number_duplicates # {2: 1802} all duplicates are based on two similar sequences (901 sequences)
# create dictionary with sequences as keys and list of headers with that sequence as values


In [ ]:
# check if duplicated_sequences_dict headers have the same label => 2 time no
for key, value in duplicated_sequences_dict.items():
    if len(value) == 2:
        if value[0].split(':')[0] == value[1].split(':')[0]:
            continue
        else:
            print("different labels")
            print(value[0].split(':')[0])
            print(value[1].split(':')[0])
            print(value)

In [ ]:
# check if all headers in header2 have _revc in the name yes and non has _forw in its name
count_revc_in_first_header = 0
count_forw_in_second_header = 0
for key, value in duplicated_sequences_dict.items():
    # check if same label
    if len(value) == 2:
        if "_revc" not in value[0]:
             count_revc_in_first_header += 1
        if "_forw" not in value[1]:
            count_forw_in_second_header += 1
        
print(count_revc_in_first_header) # 1802
print(count_forw_in_second_header) # 1802

In [ ]:
# merge the headers to one header
# ### implemented logic: 
# if same label, write the label in the beginning and merge the second header without the label add _revcomp to the end of the second header
# if not the same label, write both headers with the label and add _revcomp to the end of the second header
merged_headers = merge_duplicated_headers(duplicated_sequences_dict)
merged_headers
# header_list = []
# for key, value in duplicated_sequences_dict.items():
#     # check if same label
#     if len(value) == 2:
#         # if same label, merge headers
#         if value[0].split(':')[0] == value[1].split(':')[0]:
#             label=value[0].split(':')[0]
#             new_header=label + ":" + "_".join(":".join(value[0].split(':')[1:]).split("_")[:-1]) + "_" + "_".join(":".join(value[0].split(':')[1:]).split("_")[:-1]) + "_revcomp" # add _revcomp to the end of the second header
#         else: # different labels
#             new_header="_".join(value[0].split("_")[:-1]) + "_" + "_".join(value[1].split("_")[:-1]) + "_revcomp" # add _revcomp to the end of the second header
#     # add the new header
#     header_list.append(new_header)


In [ ]:
duplicated_sequences_dict
# example: cardiac_neuro_cava_random:NOS3|ENSG00000164867.11|EH38E3808220_fwd_tile1-1_forw; >cardiac_neuro_cava_random:KCNH2|ENSG00000055118.17|EH38E3808220_rev_tile1-1_revc
# AGCGAGAGCAGGACAGGGGCCACCAAGGGGAGGCACCAAGGTGGGAAGTAGAGAAAGGCATTCTCTGAGAGCAGGAGCCAAACAAGAGAGCTGGGAGCAGGGAAAACCCTAGGCCCCTGTCTCTTCCCGAGGGAATGTGGCCGCTGAGCCCAACCCAACCTGGAACAAGCAGTCTCTGTGTTGGGACAGAAGCTGGGACAGAAAAAGAGACAAGATTCCTTCCCGCAATCCAGAAAGAACTCGGGGACCTAGAAACAGAGGCAGGCTGGC (270bp)


In [ ]:
# make table from duplicated_sequences_dict make two columns (header1  and header2 out of values) and one sequence column
duplicated_sequences_tbl = pd.DataFrame.from_dict(duplicated_sequences_dict, orient='index').reset_index().rename(columns={'index':'sequence'})
duplicated_sequences_tbl.columns = ['sequence', 'header1', 'header2']
duplicated_sequences_tbl.head()

# write to file
duplicated_sequences_tbl.to_csv(config["files"]["duplicated_sequence_tbl"], sep='\t', index=False, header=False)

#### In case the reverse complement didn't work in bash
- load design fasta 
- generate reverse complement